In [ ]:
import pandas as pd

# === FILE PATHS ===
mapping_file = "D:/Tushar/main_with_subs_only.xlsx"
indent_file  = "D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

# === LOAD FILES ===
print("Loading mapping file...")
df_mapping = pd.read_excel(mapping_file)

print("Loading Monthly Indent file - Sheet1...")
df_indent = pd.read_excel(indent_file, sheet_name="Sheet1")

# ── VERY IMPORTANT: SHOW ACTUAL COLUMN NAMES ──
print("\n" + "="*60)
print("COLUMNS IN main_with_subs_only.xlsx:")
print(df_mapping.columns.tolist())
print("\nCOLUMNS IN Monthly Indent.xlsx (Sheet1):")
print(df_indent.columns.tolist())
print("="*60 + "\n")

# Preview first few rows of indent file (helps see headers)
print("First 3 rows of Monthly Indent Sheet1:")
print(df_indent.head(3))
print("\n")

# === RENAME COLUMNS IN MAPPING FILE ===
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',
    'Sub_Label':  'Switch_Part',
    'Main_Count': 'Qty_per_Switch',
    'Sub_Count':  'Historical_Total'  # optional
})

# === RENAME THE SWITCH / PART COLUMN IN INDENT FILE ===
# ↓↓↓ CHANGE THIS LINE after seeing the printed columns above ↓↓↓
# Examples:
# df_indent = df_indent.rename(columns={'Part No.': 'Switch_Part'})
# df_indent = df_indent.rename(columns={'Part Number': 'Switch_Part'})
# df_indent = df_indent.rename(columns={'PART NUMBER': 'Switch_Part'})
# df_indent = df_indent.rename(columns={'Item Code': 'Switch_Part'})

# ←←← Put the REAL column name here (exact match, including spaces/case)
df_indent = df_indent.rename(columns={'Part number': 'Switch_Part'})   # ← EDIT THIS

# If rename failed, you'll see NaN later — check printed columns!

# === DETECT MONTH COLUMNS AUTOMATICALLY ===
# Looks for columns containing '26' or apostrophe
month_cols = [col for col in df_indent.columns 
              if '26' in col and col != 'Switch_Part']

print("Detected month columns:", month_cols)
if not month_cols:
    print("WARNING: No month columns detected. Check printed columns above.")

# Create clean versions (Feb'26 → Feb26)
clean_months = [m.replace("'", "").replace(" ", "") for m in month_cols]

# === MERGE ===
print("\nPerforming merge...")
df_merged = pd.merge(
    df_mapping[['Child_Part', 'Switch_Part', 'Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

# Show how many matches we got
print(f"Rows after merge: {len(df_merged)}")
print(f"Rows with missing monthly data: {df_merged[month_cols[0]].isna().sum() if month_cols else 'N/A'} (if high → column name mismatch)")

# === CALCULATE DAILY & 2-DAYS ===
for month, clean in zip(month_cols, clean_months):
    daily_col   = f"Daily_{clean}"
    twodays_col = f"2Days_{clean}"
    
    df_merged[daily_col]   = (df_merged[month] / 30.0).round(2)
    df_merged[twodays_col] = (df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2).round(2)

# ─────────────────────────────────────────────
# PART 1: TOTALS PER CHILD
# ─────────────────────────────────────────────
agg_dict = {f"Daily_{clean}": 'sum' for clean in clean_months}

totals = df_merged.groupby('Child_Part', as_index=False).agg(agg_dict)

for clean in clean_months:
    totals[f"2Days_{clean}"] = (totals[f"Daily_{clean}"] * 2).round(2)

totals_cols = (
    ['Child_Part'] +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)

totals = totals[totals_cols]

totals.to_excel("Child_Totals_2Days_Per_Month.xlsx", index=False)
print(f"Totals file created: Child_Totals_2Days_Per_Month.xlsx ({len(totals)} rows)")

# ─────────────────────────────────────────────
# PART 2: DETAILED BREAKDOWN
# ─────────────────────────────────────────────
detailed_cols = (
    ['Child_Part', 'Switch_Part', 'Qty_per_Switch'] +
    month_cols +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)

detailed = df_merged[detailed_cols]

detailed.to_excel("Child_Detailed_Breakdown_2Days.xlsx", index=False)
print(f"Detailed file created: Child_Detailed_Breakdown_2Days.xlsx ({len(detailed)} rows)")

print("\nFINISHED — check the two new Excel files in your folder.")